<img src="https://github.com/milatovicdanka90-maker/DSB-Unit-03/blob/main/labs/03%20-%20Supervised-Model-Comparison/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Lab: Supervised Learning Model Comparison
---

Recall the data science process.

1. Define the problem.
2. Gather the data.
3. Explore the data.
4. Model the data.
5. Evaluate the model.
6. Answer the problem.

In this lab, we're going to focus mostly on creating (and then comparing) many regression and classification models. Thus, we'll define the problem and gather the data for you.
Most of the questions requiring a written response can be written in 2-3 sentences.

### Step 1: Define the problem.

You are a data scientist with a financial services company. Specifically, you want to leverage data in order to identify potential customers.

If you are unfamiliar with "401(k)s" or "IRAs," these are two types of retirement accounts. Very broadly speaking:
- You can put money for retirement into both of these accounts.
- The money in these accounts gets invested and hopefully has a lot more money in it when you retire.
- These are a little different from regular bank accounts in that there are certain tax benefits to these accounts. Also, employers frequently match money that you put into a 401k.
- If you want to learn more about them, check out [this site](https://www.nerdwallet.com/article/ira-vs-401k-retirement-accounts).

We will tackle one regression problem and one classification problem today.
- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

Check out the data dictionary [here](http://fmwww.bc.edu/ec-p/data/wooldridge2k/401KSUBS.DES).

### NOTE: When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable. When predicting `e401k`, you may use the entire dataframe if you wish.

### Step 2: Gather the data.

##### 1. Read in the data from the repository.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV

import statsmodels.api as sm
import itertools

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor

In [11]:
df_e = pd.read_csv('https://raw.githubusercontent.com/milatovicdanka90-maker/DSB-Unit-03/refs/heads/main/labs/03%20-%20Supervised-Model-Comparison/data/401ksubs.csv')

In [12]:
df_e.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9275 entries, 0 to 9274
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   e401k   9275 non-null   int64  
 1   inc     9275 non-null   float64
 2   marr    9275 non-null   int64  
 3   male    9275 non-null   int64  
 4   age     9275 non-null   int64  
 5   fsize   9275 non-null   int64  
 6   nettfa  9275 non-null   float64
 7   p401k   9275 non-null   int64  
 8   pira    9275 non-null   int64  
 9   incsq   9275 non-null   float64
 10  agesq   9275 non-null   int64  
dtypes: float64(3), int64(8)
memory usage: 797.2 KB


In [13]:
df_e.head()

,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
0,0,13.170,0,0,40,1,4.575,0,1,173.4489,1600
1,1,61.230,0,1,35,1,154.000,1,0,3749.1130,1225
2,0,12.858,1,0,44,2,0.000,0,0,165.3282,1936
3,0,98.880,1,1,44,2,21.800,0,0,9777.2540,1936
4,0,22.614,0,0,53,1,18.450,0,0,511.3930,2809


In [14]:
df_e.isnull().sum()

,0
e401k,0
inc,0
marr,0
male,0
age,0
fsize,0
nettfa,0
p401k,0
pira,0
incsq,0


In [15]:
df_e.describe()

,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
count,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000
mean,0.392129,39.254641,0.628571,0.204420,41.080216,2.885067,19.071675,0.276226,0.254340,2121.192483,1793.652722
std,0.488252,24.090002,0.483213,0.403299,10.299517,1.525835,63.963838,0.447154,0.435513,3001.469424,895.648841
min,0.000000,10.008000,0.000000,0.000000,25.000000,1.000000,-502.302000,0.000000,0.000000,100.160100,625.000000
25%,0.000000,21.660000,0.000000,0.000000,33.000000,2.000000,-0.500000,0.000000,0.000000,469.155600,1089.000000
50%,0.000000,33.288000,1.000000,0.000000,40.000000,3.000000,2.000000,0.000000,0.000000,1108.091000,1600.000000
75%,1.000000,50.160000,1.000000,0.000000,48.000000,4.000000,18.449500,1.000000,1.000000,2516.025500,2304.000000
max,1.000000,199.041000,1.000000,1.000000,64.000000,13.000000,1536.798000,1.000000,1.000000,39617.320000,4096.000000


##### 2. What are 2-3 other variables that, if available, would be helpful to have?

In [ ]:
#Education level, occupation (industry) and current debt(credit score)

##### 3. Suppose a peer recommended putting `race` into your model in order to better predict who to target when advertising IRAs and 401(k)s. Why would this be an unethical decision?

In [ ]:
# Including race in the model would be unethical because it introduces algorithmic bias and risks institutionalizing systemic discrimination.
# If the model learns from historical socioeconomic disparities, it might stop targeting minority groups with retirement advertisements,
# further widening the wealth gap. Furthermore, using race as a determining factor for financial product targeting violates strict fair lending
# and anti-discrimination regulations.

## Step 3: Explore the data.

##### 4. When attempting to predict income, which feature(s) would we reasonably not use? Why?

In [ ]:
# When predicting income, we should not use e401k (401k eligibility), p401k (401k participation), and pira (IRA participation).
# These variables are direct outcomes of a person's financial choices and are heavily dependent on having a stable or higher income
# in the first place. Including them would cause data leakage, making the model artificially accurate during training but unrealistic
# for predicting the income of new, unseen customers.

##### 5. What two variables have already been created for us through feature engineering? Come up with a hypothesis as to why subject-matter experts may have done this.
> This need not be a "statistical hypothesis." Just brainstorm why SMEs might have done this!

In [ ]:
# The two variables created through feature engineering are incsq (income squared) and agesq (age squared).
# Subject-matter experts created them because financial behaviors like earning and saving do not grow in a straight, simple line as you get older.
# These squared variables help the model capture non-linear patterns, such as income growing quickly in your youth, peaking in middle age,
# and dropping after retirement.

##### 6. Looking at the data dictionary, one variable description appears to be an error. What is this error, and what do you think the correct value would be?

The error is in the description of male, which is defined in the data dictionary as a boolean variable (1 for male, 0 for female). However, looking at the dataset's summary statistics, the mean for male is 0.204, meaning only about 20% of the sample is male. The mistake is that the column actually represents female (where 1 means female and 0 means male), or it is simply heavily skewed towards females in this specific dataset.
Message Jelena Vasic

## Step 4: Model the data. (Part 1: Regression Problem)

Recall:
- Problem: What features best predict one's income?
- When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable.

##### 7. List all modeling tactics we've learned that could be used to solve a regression problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific regression problem and explain why or why not.

• Simple Linear Regression (SLR): No. It only uses one feature, which is too simple to predict a complex topic like income.

• Multiple Linear Regression (MLR / OLS): Yes. It allows us to use all valid features together and gives us a clear baseline model.

• Ridge Regression (\(L_{2}\)): Yes. It helps prevent overfitting and handles the multicollinearity caused by our squared features (agesq).

• LASSO Regression (\(L_{1}\)): Yes. It is great for feature selection because it can drop useless features by shrinking their coefficients to zero.

• \(k\)-Nearest Neighbors (KNN) Regressor: Yes. It can find complex, non-linear patterns, but we must scale our data first.

• Logistic Regression: No. This is a classification algorithm used for predicting categories (like Yes/No), so it cannot predict a continuous number like income.

##### 8. Regardless of your answer to number 7, fit at least one of each of the following models to attempt to solve the regression problem above:
    - a multiple linear regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend setting a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [16]:
# 1. Definisanje cilja (y) i osobina (X) uz izbacivanje zabranjenih kolona
target = 'inc'
forbidden_cols = ['inc', 'e401k', 'p401k', 'pira']
features = [col for col in df_e.columns if col not in forbidden_cols]

X = df_e[features]
y = df_e[target]

# 2. Podela na trening i test skup (uz fiksiran random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Kreiranje i treniranje svih 6 traženih modela
# Napomena: KNN zahteva skaliranje, pa koristimo Pipeline za njega
models = {
    "Multiple Linear Regression": LinearRegression(),
    "k-Nearest Neighbors": Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())]),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Bagged Decision Trees": BaggingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42)
}

# Treniranje svakog modela i štampanje R² rezultata za brzu proveru
for name, model in models.items():
    model.fit(X_train, y_train)
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f"{name}: Train R² = {train_score:.3f} | Test R² = {test_score:.3f}")

Multiple Linear Regression: Train R² = 0.896 | Test R² = 0.903
k-Nearest Neighbors: Train R² = 0.981 | Test R² = 0.974
Decision Tree: Train R² = 1.000 | Test R² = 1.000
Bagged Decision Trees: Train R² = 1.000 | Test R² = 1.000
Random Forest: Train R² = 1.000 | Test R² = 1.000
AdaBoost: Train R² = 0.992 | Test R² = 0.992


##### 9. What is bootstrapping?

##### 10. What is the difference between a decision tree and a set of bagged decision trees? Be specific and precise!

##### 11. What is the difference between a set of bagged decision trees and a random forest? Be specific and precise!

##### 12. Why might a random forest be superior to a set of bagged decision trees?
> Hint: Consider the bias-variance tradeoff.

## Step 5: Evaluate the model. (Part 1: Regression Problem)

##### 13. Using RMSE, evaluate each of the models you fit on both the training and testing data.

##### 14. Based on training RMSE and testing RMSE, is there evidence of overfitting in any of your models? Which ones?

##### 15. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

##### 16. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

## Step 4: Model the data. (Part 2: Classification Problem)

Recall:
- Problem: Predict whether or not one is eligible for a 401k.
- When predicting `e401k`, you may use the entire dataframe if you wish.

##### 17. While you're allowed to use every variable in your dataframe, mention at least one disadvantage of using `p401k` in your model.

##### 18. List all modeling tactics we've learned that could be used to solve a classification problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific classification problem and explain why or why not.

##### 19. Regardless of your answer to number 18, fit at least one of each of the following models to attempt to solve the classification problem above:
    - a logistic regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend using a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

## Step 5: Evaluate the model. (Part 2: Classfication Problem)

##### 20. Suppose our "positive" class is that someone is eligible for a 401(k). What are our false positives? What are our false negatives?

##### 21. In this specific case, would we rather minimize false positives or minimize false negatives? Defend your choice.

##### 22. Suppose we wanted to optimize for the answer you provided in problem 21. Which metric would we optimize in this case?

##### 23. Suppose that instead of optimizing for the metric in problem 21, we wanted to balance our false positives and false negatives using `f1-score`. Why might [f1-score](https://en.wikipedia.org/wiki/F1_score) be an appropriate metric to use here?

##### 24. Using f1-score, evaluate each of the models you fit on both the training and testing data.

##### 25. Based on training f1-score and testing f1-score, is there evidence of overfitting in any of your models? Which ones?

##### 26. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

##### 27. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

## Step 6: Answer the problem.

##### BONUS: Briefly summarize your answers to the regression and classification problems. Be sure to include any limitations or hesitations in your answer.

- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.